## Module 1: Agentic RAG

This module consists of building a simple RAG pipeline using keyword search, and making it agentic, so the LLM decides when and what to search instead of running a fixed pipeline.

### Part 2: Agents

This section puts the LLM in charge of the search decisions, turning the fixed pipeline into an agent. Standard, rigid RAG pipelines fail when lexical keyword search misses relevant documents due to typos, unusual phrasing, or the need for multiple queries. To overcome this limitation, we can transition to an agentic system by handing control to the LLM instead of using a fixed flow. This allows the LLM to dynamically drive the process by fixing typos, trying alternative search terms, or asking clarifying questions. Ultimately, an agent leverages the LLM to flexibly decide which tools to use and in what order through an iterative agentic loop.

- With RAG, the developer decides. The search always runs once with the exact user query.
- With an agent, the LLM decides. It chooses which actions to take and when to stop.

In [3]:
# load required libraries
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI

# start the OpenAI client
openai_client = OpenAI()

- **Asking without tools:**

Without any tools, the model answers from its general knowledge in a vague and not helpful manner.

In [4]:
# send a message to LLM and observe the output
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)
print(response.output_text)

Absolutely — in most cases, yes, you can still join a course after it has started or after you’ve discovered it late.

A few things to check:
- **Enrollment deadline**: Some courses have a cutoff date.
- **Capacity**: The class may already be full.
- **Prerequisites**: You may need to meet certain requirements first.
- **Catch-up work**: If it’s already in progress, you may need to make up missed lessons.

If you want, I can help you draft a quick message to the instructor or the course organizer asking if late enrollment is possible.


- **Defining and asking with tools:**

When a tool is defined, the model can reference it by its name and use it to improve the results.

In [7]:
from ingest import load_faq_data, build_index

# load data and build index
documents = load_faq_data()
index = build_index(documents)

In [8]:
# define a top-level search function that queries the index directly
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:
# use the search function to observe the output
search("How do I run Ollama?")

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

The model does not see the code, only a schema describing what the function does and what arguments it takes. LLMs are language agnostic. At the end we're just making an HTTP call, so we describe the tool in JSON rather than in Python. The same schema would work from TypeScript or Java. The `description` is the most important field, because the model reads it to decide when to call the function. `parameters` is a JSON schema for the arguments, and we mark query as required so the model always fills it in.

In [ ]:
# define the tool
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [11]:
# send the same question with the tool
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered recently can I join enrollment eligibility late join"}', call_id='call_lrY94Hh44MNZ5WUvM1r3UZpA', name='search', type='function_call', id='fc_07ef7bb4dd419055006a31432ce67c81a1aa4405d057974327', namespace=None, status='completed')]

Instead of a message with the answer, the response contains a `function_call` entry. The model decided it needs to search the FAQ before answering. Rather than reply, it asked to run the search function first.

As it seems in the arguments, the model did not pass the question verbatim. It judged the raw question was not the best query to search with, and rewrote the question into search keywords.

- **Executing the function and sending the results back:**

As it seems from the response output, LLM changed the query and asked for a function call with the arguments it defined.  

In [16]:
print(f"LLM call type: {response.output[0].type}")
print(f"LLM call name: {response.output[0].name}")
print(f"LLM call arguments: {response.output[0].arguments}")

LLM call type: function_call
LLM call name: search
LLM call arguments: {"query":"join course discovered recently can I join enrollment eligibility late join"}


Hence the results need to be parsed to call the `search` function and serialized before being sent back to the model.

In [ ]:
import json

# parse the response
call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

Because LLMs are entirely stateless between API calls, they possess no memory unless the entire interaction history is replayed in the input list. Consequently, the function-calling loop for "agentic RAG" requires multiple round-trips to send the original question, the tool-use decision, and the search results back to the model. This iterative pattern allows the LLM to maintain context and dynamically decide which tools to call to produce a proper, accurate answer. Hence, send both the previous history and the search result to the LLM as below: 

In [18]:
# add model's output to the conversation history
messages.extend(response.output)

# add the result from the tool
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [20]:
# the extended messages which will be sent to the LLM
print(messages)

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}, ResponseFunctionToolCall(arguments='{"query":"join course discovered recently can I join enrollment eligibility late join"}', call_id='call_lrY94Hh44MNZ5WUvM1r3UZpA', name='search', type='function_call', id='fc_07ef7bb4dd419055006a31432ce67c81a1aa4405d057974327', namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_lrY94Hh44MNZ5WUvM1r3UZpA', 'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a ce

- **Ask the model again with results:**

As it seems from the response output, LLM changed the query and asked for a function call with the arguments it defined.  

In [21]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)
print(response.output_text)

Yes — you can still join and start learning anytime.

If you want a certificate, the key thing is to submit your project while submissions are still open.


- **Token usage and costs:**

Since there are more than one API calls, it is better to check the one tool-using turn cost. The usage below is only for the second API call. The first call also has its own usage and its own cost, which was the call where the model decided to invoke search. Two calls means paying twice, and the second call costs even more due to resending the full history as input.

With a real agent loop the model can make many calls and the costs add up. So it is better to keep an eye on usage while developing.

In [22]:
usage = response.usage
print(usage.input_tokens)
print(usage.output_tokens)

772
35


In [23]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(772, 35)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001368


- **The Agentic Loop:**

So far the loop with an agent was as such:

1. Make a call to the LLM (1st API call)
2. LLM decided to invoke a search (with params)
3. Invoke the search and get the results
4. Send the results back to the LLM (2nd API call)
5. LLM processes the results
6. LLM gives the answer to the initial question

However, LLM may decide to make more API calls depending on the results. In such case, a loop should be built to allow LLM to make as many calls as possible until a satisfying answer is obtained. To do so, LLM instructions should be revisited.

In [27]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

Then wrap the function calling into a function to allow multiple function calls.

In [25]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    # add any other function calls if necessary
    
    result_json = json.dumps(results, indent=2)

    return { # return the part to be sent as additional message to LLM
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json
    }

Then process a single model response by appending each output entry to the conversation, printing any messages, and running any function calls. Function-call results get appended too.

In [28]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course enroll discovered course can I join"}
function_call: search {"query":"course enrollment registration late join discovered course"}


Currently the LLM does not have an answer and only function calls, hence the search results need to be sent back to the LLM. It should be done in a loop to allow consequent calls.

In [29]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course enroll discovered course can I join"}', call_id='call_IV3lADnr7kFb2MwEa6j36kC2', name='search', type='function_call', id='fc_037869b7699f2e9f006a3156ac5c78819e9f4ebdf01b56790b', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"course enrollment registration late join discovered course"}', call_id='call_WIqyAGDuTFyfVkW5T2t

In [31]:
# if necessary, update instructions to ask the model not stop and make more calls
instructions_multiple = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results and then perform more searches.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [30]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course late enrollment discovered course can I join"}
function_call: search {"query":"course enrollment after start discovered course can I join"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, the key condition is that you need to submit your project while submissions are still open. Also, certificates are only available for the live cohort, not self-paced participation.

If you want, I can also help you with how to start the course or what the weekly workflow looks like.


Finally wrap all into an agentic loop function to call easily.

In [32]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [33]:
# test with a typo
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run install local model Ollama FAQ"}
iteration #2...
function_call: search {"query":"Ollama local run install FAQ local model server llama2 docker"}
iteration #3...
ASSISTANT:
To run **Ollama** locally, the basic flow is:

1. **Install Ollama**
   - macOS / Windows: download from the Ollama website and install it
   - Linux: use the install script from Ollama’s docs

2. **Start the service**
   - On many systems, Ollama runs in the background automatically after install.
   - If needed, start it manually.

3. **Pull a model**
   ```bash
   ollama pull llama3.1
   ```

4. **Run the model**
   ```bash
   ollama run llama3.1
   ```

5. **Use the local API**
   - By default, Ollama serves on:
     ```bash
     http://localhost:11434
     ```

   - Example request:
     ```bash
     curl http://localhost:11434/api/generate -d '{
       "model": "llama3.1",
       "prompt": "Hello!"
     }'
     ```

If you meant **“Olama”** as a 

'To run **Ollama** locally, the basic flow is:\n\n1. **Install Ollama**\n   - macOS / Windows: download from the Ollama website and install it\n   - Linux: use the install script from Ollama’s docs\n\n2. **Start the service**\n   - On many systems, Ollama runs in the background automatically after install.\n   - If needed, start it manually.\n\n3. **Pull a model**\n   ```bash\n   ollama pull llama3.1\n   ```\n\n4. **Run the model**\n   ```bash\n   ollama run llama3.1\n   ```\n\n5. **Use the local API**\n   - By default, Ollama serves on:\n     ```bash\n     http://localhost:11434\n     ```\n\n   - Example request:\n     ```bash\n     curl http://localhost:11434/api/generate -d \'{\n       "model": "llama3.1",\n       "prompt": "Hello!"\n     }\'\n     ```\n\nIf you meant **“Olama”** as a different tool, tell me the exact name and I’ll tailor the steps.\n\nWould you like a quick setup guide for **macOS, Windows, or Linux**?'

In [34]:
# test with the usual question
agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join course discovered course can I join enroll late enrollment FAQ"}
function_call: search {"query":"course enrollment late join discovered course FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, make sure to submit your project while submissions are still open. Also, if the course is running live, that’s when certificates and peer review are available.

If you want, I can also help you figure out how to start the course or whether it’s still possible to get a certificate.


'Yes — you can still join the course.\n\nIf you want a certificate, make sure to submit your project while submissions are still open. Also, if the course is running live, that’s when certificates and peer review are available.\n\nIf you want, I can also help you figure out how to start the course or whether it’s still possible to get a certificate.'

In [35]:
# test with multiple call instructions
agent_loop(instructions_multiple, question)

iteration #1...
function_call: search {"query":"join the course enroll late registration discovered the course can I join"}
iteration #2...
function_call: search {"query":"certificate project accepting submissions self-paced live cohort register accepted start learning submitting homework while form is open"}
iteration #3...
ASSISTANT:
Yes — you can join the course even if you just discovered it.

A couple of important notes:
- You can start learning and submitting homework while the course form is open.
- If you want a certificate, you need to submit your project while submissions are still being accepted.
- Certificates are only awarded for finishing with a live cohort, not in self-paced mode.

If you want, I can also help you with how to start the course or explain the weekly workflow.


'Yes — you can join the course even if you just discovered it.\n\nA couple of important notes:\n- You can start learning and submitting homework while the course form is open.\n- If you want a certificate, you need to submit your project while submissions are still being accepted.\n- Certificates are only awarded for finishing with a live cohort, not in self-paced mode.\n\nIf you want, I can also help you with how to start the course or explain the weekly workflow.'


- **Using Frameworks:**

Handwritten agent loops are educational but become highly repetitive because one must manually rewrite the same control loops, function-handling logic, and message management for every new system. To eliminate this boilerplate, orchestration frameworks wrap these patterns, allowing developers to focus entirely on defining tools, prompts, and system behaviors. While these production-grade frameworks abstract the underlying logic, they use the exact same looping structures under the hood. 

For learning and local debugging use ToyAIKit framework, which is strictly a non-production teaching aid.